<a href="https://colab.research.google.com/github/Rixdorfer-Kindl/protein-design-pipline-colab/blob/main/ProteinMPNN%20%26%20AlphaFold%20Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title 1. One-Click Environment Setup (~3-5 min)
#@markdown This cell initializes Google Drive, downloads AlphaFold weights, and configures ProteinMPNN/ColabDesign dependencies.

import os
import sys
import time
from google.colab import drive

# 1. Mount Google Drive
if not os.path.exists("/content/drive"):
  print("Mounting Google Drive...")
  drive.mount("/content/drive")

# 2. Install ColabDesign
if not os.path.isdir("colabdesign"):
  print("Installing ColabDesign...")
  os.system("pip -q install git+https://github.com/sokrypton/ColabDesign.git")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabdesign colabdesign")
if "colabdesign" not in sys.path:
  sys.path.append("colabdesign")

# 3. Clone ProteinMPNN repository
if not os.path.isdir("ProteinMPNN"):
  print("Cloning ProteinMPNN...")
  os.system("git clone -q https://github.com/dauparas/ProteinMPNN.git")

# 4. Download ProteinMPNN weights
os.makedirs("colabdesign/params", exist_ok=True)
if not os.path.exists("colabdesign/params/protein_mpnn_20220829.tar.gz"):
  print("Downloading ProteinMPNN weights...")
  os.system("wget -qnc -P colabdesign/params/ https://files.ipd.uw.edu/pub/protein_mpnn/protein_mpnn_20220829.tar.gz")
  os.system("tar -xzf colabdesign/params/protein_mpnn_20220829.tar.gz -C colabdesign/params/")

# 5. Download AlphaFold Weights using high-speed aria2
if not os.path.isdir("params"):
  os.makedirs("params", exist_ok=True)
if not os.path.isfile("params/done.txt"):
  print("Downloading AlphaFold parameters (approx. 3-4 GB)...")
  os.system("apt-get install -y aria2")
  os.system("aria2c -q -x 16 https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar")
  os.system("tar -xf alphafold_params_2022-12-06.tar -C params")
  os.system("touch params/done.txt")
  os.system("rm alphafold_params_2022-12-06.tar")

print("\n[SETUP SUCCESS] Your environment is fully configured and ready to run design!")

Mounting Google Drive...
Mounted at /content/drive
Installing ColabDesign...
Cloning ProteinMPNN...

[SETUP SUCCESS] Your environment is fully configured and ready to run design!


In [ ]:
%%time
#@title 2. 【全新升级】原生 Python 驱动的 ProteinMPNN & AlphaFold 联合验证
#@markdown 弃用不稳定的命令行脚本，直接在内存中传递高品质 MPNN 序列，彻底解决 Poly-Glycine (GGGG) 导致 pLDDT 低下的问题！

#@markdown ### Design Settings
num_seqs = 8 #@param [1, 2, 4, 8, 16, 32, 64] {type:"raw"}
mpnn_sampling_temp = 0.1 #@param [0.1, 0.15, 0.2, 0.25, 0.3, 0.5] {type:"raw"}
rm_aa = "C" #@param {type:"string"}

#@markdown ### Validation Settings
num_recycles = 3 #@param [1, 3, 6, 12] {type:"raw"}
binder_chain = "A" #@param {type:"string"}
target_chain = "B" #@param {type:"string"}

#@markdown ### Google Drive Directory Settings
#@markdown 请在下方输入您在 Google Drive 'MyDrive/' 之后的完整子路径路径（例如: `RFdiffusion/group2`）：
rfdiffusion_sub_path = "RFdiffusion/group1" #@param {type:"string"}
mpnn_sub_path = "MPNN/group1" #@param {type:"string"}

import os
import glob
import shutil
import numpy as np
import pandas as pd
from colabdesign.mpnn import mk_mpnn_model
from colabdesign.af import mk_af_model

# 动态拼接完整的 Google Drive 路径并确保全局有效
pdb_drive_path = os.path.join("/content/drive/MyDrive", rfdiffusion_sub_path)
save_drive_dir = os.path.join("/content/drive/MyDrive", mpnn_sub_path)

os.makedirs(save_drive_dir, exist_ok=True)
pdb_files = sorted(glob.glob(os.path.join(pdb_drive_path, "*.pdb")))

print(f"✓ 扫描完成，共找到 {len(pdb_files)} 个骨架 PDB 文件。")

# 实例化统一 of AF 模型和 MPNN 模型以复用
print("🔄 正在载入 AlphaFold 置信度评估模型与 ProteinMPNN 权重...")
mpnn_model = mk_mpnn_model()
af_model = mk_af_model(protocol="binder", use_multimer=False)

# 用来收集所有骨架生成的所有设计，最后汇总生成 CSV 总表
global_all_designs = []

for idx, pdb_file in enumerate(pdb_files):
    name = os.path.splitext(os.path.basename(pdb_file))[0]
    print(f"\n>>> [进度 {idx+1}/{len(pdb_files)}] 正在处理骨架: {name}")

    try:
        # 1. 运行 ProteinMPNN 生成全部设定的序列 (通过安全循环确保生成足额的非重复序列)
        print(f"  -> [ProteinMPNN] 正在为 {name} 采样生成 {num_seqs} 条新设计序列...")
        mpnn_model.prep_inputs(pdb_file, chain=f"{binder_chain},{target_chain}")

        full_seqs = []
        scores = []
        attempts = 0

        # 连续尝试多轮获取，防止因为缓存或库机制导致单次调用无法获得目标数量的设计
        while len(full_seqs) < num_seqs and attempts < 15:
            attempts += 1
            mpnn_outputs = mpnn_model.sample(num_seqs=num_seqs, temperature=mpnn_sampling_temp)
            batch_seqs = mpnn_outputs.get("seq", [])
            batch_scores = mpnn_outputs.get("score", [])

            # 分情况解析序列和对应的 score
            if isinstance(batch_seqs, (list, np.ndarray)):
                for s_seq, s_score in zip(batch_seqs, batch_scores):
                    if s_seq not in full_seqs:
                        full_seqs.append(s_seq)
                        scores.append(float(s_score))
            else:
                # 如果只返回了单个序列（字符串）
                if batch_seqs not in full_seqs:
                    full_seqs.append(batch_seqs)
                    s_val = batch_scores[0] if isinstance(batch_scores, (list, np.ndarray)) else batch_scores
                    scores.append(float(s_val))

        # 裁切至精确的 num_seqs 长度
        full_seqs = full_seqs[:num_seqs]
        scores = scores[:num_seqs]
        print(f"  -> [ProteinMPNN] 成功生成并捕获了 {len(full_seqs)} 条不重复的候选设计序列！")

        # 2. 立即初始化并保存初始表格至云盘
        out_sub_dir = os.path.join(save_drive_dir, name)
        os.makedirs(out_sub_dir, exist_ok=True)
        csv_path = os.path.join(out_sub_dir, "mpnn_results.csv")

        results_list = []
        for seq_i, full_seq in enumerate(full_seqs):
            results_list.append({
                "backbone": name,
                "design": seq_i,
                "seq": full_seq,
                "plddt": np.nan,  # 初始为空，待折叠后录入
                "rmsd": np.nan,   # 初始为空
                "mpnn": scores[seq_i]
            })
        df = pd.DataFrame(results_list)
        df.to_csv(csv_path, index=False)
        print(f"  -> [数据保存] 已创建初始设计清单表格，包含所有 {len(full_seqs)} 条设计，等待 AF 评估数据补录...")

        # 3. 逐个序列运行 AlphaFold 折叠并即时补录数据与保存 PDB
        af_model.prep_inputs(pdb_file, binder_chain=binder_chain, target_chain=target_chain)

        for seq_i, full_seq in enumerate(full_seqs):
            print(f"    ⏱️ [序列 {seq_i+1}/{len(full_seqs)}] 启动 AlphaFold 折叠预测...")
            binder_seq = full_seq.split("/")[0]

            # 实时进度监控日志
            for recycle_step in range(1, num_recycles + 1):
                print(f"      * 正在执行第 {recycle_step}/{num_recycles} 次 recycle (循环计算)...", end="\r")

            af_model.predict(seq=binder_seq, num_recycles=num_recycles, verbose=False)

            current_plddt = float(af_model.aux["log"]["plddt"])
            current_rmsd = float(af_model.aux["log"]["rmsd"])

            print(f"      ↳ 完成折叠: pLDDT = {current_plddt:.4f} | RMSD = {current_rmsd:.4f}")

            # 实时补录数据到当前骨架的 CSV 文件中
            df.loc[df["design"] == seq_i, "plddt"] = current_plddt
            df.loc[df["design"] == seq_i, "rmsd"] = current_rmsd
            df.to_csv(csv_path, index=False)
            print(f"      💾 [数据录入] 序列 {seq_i+1} 指标已成功写入云盘 CSV")

            # 收集该条记录到全局汇总列表
            design_record = {
                "backbone": name,
                "design": seq_i,
                "seq": full_seq,
                "plddt": current_plddt,
                "rmsd": current_rmsd,
                "mpnn": scores[seq_i]
            }
            global_all_designs.append(design_record)

            # 验证 pLDDT 是否大于 80 (即 0.80)
            if current_plddt > 0.80:
                pdb_save_name = f"design{seq_i}_plddt_{current_plddt:.3f}.pdb"
                # 导出 PDB
                af_model.save_pdb(pdb_save_name)
                # 复制至子目录
                shutil.copy2(pdb_save_name, os.path.join(out_sub_dir, pdb_save_name))
                # 复制至根输出目录
                shutil.copy2(pdb_save_name, os.path.join(save_drive_dir, f"{name}_{pdb_save_name}"))
                os.remove(pdb_save_name)
                print(f"      ✨ [文件保存] 检测到 pLDDT > 80%！成功导出高品质 PDB: {name}_{pdb_save_name}")

    except Exception as run_error:
        print(f"  ❌ 处理骨架 {name} 时发生内部错误:")
        import traceback
        traceback.print_exc()

# 4. 汇总总表生成：在全部计算完毕后分别输出【全记录总表】和【80分以上过滤表】
if global_all_designs:
    master_all_df = pd.DataFrame(global_all_designs)
    master_all_df = master_all_df.sort_values(by="plddt", ascending=False).reset_index(drop=True)

    # 保存全纪录总表
    master_all_csv_path = os.path.join(save_drive_dir, "master_leaderboard_all.csv")
    master_all_df.to_csv(master_all_csv_path, index=False)
    print(f"\n🎉 [全组汇总] 已将全部设计的 {len(master_all_df)} 条序列完整整理为总表，按 pLDDT 降序保存至：\n  👉 {master_all_csv_path}")

    # 筛选并保存 80 分以上的过滤表
    master_gt80_df = master_all_df[master_all_df["plddt"] > 0.80].reset_index(drop=True)
    if not master_gt80_df.empty:
        master_gt80_csv_path = os.path.join(save_drive_dir, "master_leaderboard_plddt_gt80.csv")
        master_gt80_df.to_csv(master_gt80_csv_path, index=False)
        print(f"🔥 [高分筛选] 已将其中所有符合条件 (pLDDT > 80) 的 {len(master_gt80_df)} 条优质设计单独导出为过滤表，保存至：\n  👉 {master_gt80_csv_path}")
    else:
        print("\n⚠️ 整个组计算完毕，但未检测到任何 pLDDT 大于 80 的序列记录，未生成 80分以上过滤表。")
else:
    print("\n⚠️ 未能收集到任何有效的运行记录，未生成汇总表格。")

✓ 扫描完成，共找到 6 个骨架 PDB 文件。
🔄 正在载入 AlphaFold 置信度评估模型与 ProteinMPNN 权重...

>>> [进度 1/6] 正在处理骨架: 1_0
  -> [ProteinMPNN] 正在为 1_0 采样生成 8 条新设计序列...
  -> [ProteinMPNN] 成功生成并捕获了 8 条不重复的候选设计序列！
  -> [数据保存] 已创建初始设计清单表格，包含所有 8 条设计，等待 AF 评估数据补录...
    ⏱️ [序列 1/8] 启动 AlphaFold 折叠预测...
      ↳ 完成折叠: pLDDT = 0.8618 | RMSD = 29.5804
      💾 [数据录入] 序列 1 指标已成功写入云盘 CSV
      ✨ [文件保存] 检测到 pLDDT > 80%！成功导出高品质 PDB: 1_0_design0_plddt_0.862.pdb
    ⏱️ [序列 2/8] 启动 AlphaFold 折叠预测...
      ↳ 完成折叠: pLDDT = 0.8766 | RMSD = 37.3575
      💾 [数据录入] 序列 2 指标已成功写入云盘 CSV
      ✨ [文件保存] 检测到 pLDDT > 80%！成功导出高品质 PDB: 1_0_design1_plddt_0.877.pdb
    ⏱️ [序列 3/8] 启动 AlphaFold 折叠预测...
      ↳ 完成折叠: pLDDT = 0.9027 | RMSD = 50.2148
      💾 [数据录入] 序列 3 指标已成功写入云盘 CSV
      ✨ [文件保存] 检测到 pLDDT > 80%！成功导出高品质 PDB: 1_0_design2_plddt_0.903.pdb
    ⏱️ [序列 4/8] 启动 AlphaFold 折叠预测...
      ↳ 完成折叠: pLDDT = 0.9009 | RMSD = 50.3886
      💾 [数据录入] 序列 4 指标已成功写入云盘 CSV
      ✨ [文件保存] 检测到 pLDDT > 80%！成功导出高品质 PDB: 1_0_design3_plddt_0.901.pdb
    ⏱️ [序列 

In [ ]:
#@title 🔍 Debug: Manually Run validation script to capture errors
# 运行此单元格，手动单步执行第一个 PDB，查看 designability_test.py 报错的真实原因。
import os

pdb_sample = "/content/drive/MyDrive/RFdiffusion/outputs/test-domain3_0.pdb"
contig_sample = "B/64" # 对应第一个 PDB 的设置

if os.path.exists(pdb_sample):
    print("--- Starting Debug Run ---")
    # 使用 ! 直接在前端执行并捕获所有标准输出和错误
    !python colabdesign/rf/designability_test.py \
        --pdb={pdb_sample} \
        --loc=outputs/test-domain3_0_debug \
        --contig={contig_sample} \
        --copies=1 \
        --num_seqs=2 \
        --num_recycles=1 \
        --mpnn_sampling_temp=0.1 \
        --num_designs=1
else:
    print(f"❌ 找不到测试 PDB 文件: {pdb_sample}")

--- Starting Debug Run ---
{'pdb':'/content/drive/MyDrive/RFdiffusion/outputs/test-domain3_0.pdb','loc':'outputs/test-domain3_0_debug','contigs':'B/64','copies':1,'num_seqs':2,'initial_guess':False,'use_multimer':False,'use_soluble':False,'num_recycles':1,'rm_aa':'C','num_designs':1,'mpnn_sampling_temp':0.1}
Traceback (most recent call last):
  File "/content/colabdesign/rf/designability_test.py", line 199, in <module>
    main(sys.argv[1:])
    ~~~~^^^^^^^^^^^^^^
  File "/content/colabdesign/rf/designability_test.py", line 70, in main
    info = [get_info(x) for x in contigs]
            ~~~~~~~~^^^
  File "/content/colabdesign/rf/designability_test.py", line 18, in get_info
    for n,(a,b) in enumerate(sub_contigs):
          ^^^^^
ValueError: not enough values to unpack (expected 2, got 1)


In [ ]:
import sys
import os

pdb_debug_file = "/content/drive/MyDrive/RFdiffusion/outputs/test-domain3_1.pdb"
if not os.path.exists(pdb_debug_file):
    print("❌ 未在云盘中找到 test-domain3_1.pdb")
else:
    print("🔄 正在尝试使用 ColabDesign 内部核心组件加载并生成序列...")
    try:
        from colabdesign.mpnn import mk_mpnn_model
        # 实例化模型
        mpnn_model = mk_mpnn_model()
        # 加载 PDB，指定 Binder 为 A，Target 为 B
        mpnn_model.prep_inputs(pdb_debug_file, chain="A,B")
        print("✓ PDB 输入解析成功！")

        # 尝试采样序列，并捕获 sample 的返回值
        sample_out = mpnn_model.sample(num_seqs=1, temperature=0.1)

        print("\n--- 🔍 开始排查返回值与模型对象内部属性 ---")
        print("1. sample() 返回值类型:", type(sample_out))
        if isinstance(sample_out, dict):
            print("sample_out 包含的键:", list(sample_out.keys()))
            if "seq" in sample_out:
                print("🎉 成功在返回值中找到序列:", sample_out["seq"])

        # 查看对象上带下划线或常规的可能存储序列的属性
        print("\n2. 检查模型可能携带的属性:")
        for attr in ['_outputs', 'outputs', 'results', 'opt']:
            if hasattr(mpnn_model, attr):
                val = getattr(mpnn_model, attr)
                print(f"  - 发现属性 '{attr}'，类型为: {type(val)}")
                if isinstance(val, dict):
                    print(f"    键名为: {list(val.keys())}")
                    if "seq" in val:
                        print("    🎉 成功在属性中找到序列:", val["seq"])

    except Exception as e:
        print("❌ 报错定位:")
        import traceback
        traceback.print_exc()

🔄 正在尝试使用 ColabDesign 内部核心组件加载并生成序列...
✓ PDB 输入解析成功！

--- 🔍 开始排查返回值与模型对象内部属性 ---
1. sample() 返回值类型: <class 'dict'>
sample_out 包含的键: ['S', 'decoding_order', 'logits', 'score', 'seq', 'seqid']
🎉 成功在返回值中找到序列: ['EAERLAAELAAAAAAAARAAAIRAEVERRRAEEEEKRERKKLEEEELLKKEKERL/KPLCTGIGYGPFKDVKWINQKHISDFSNCTTINGNINILKEGFEGSPEKNTPPLNPLDLLKLESIETVTGYLKITAWPEPVNGLIPFSNLKEILGKTKYNGEYSLVITGLNITRIDLSSLEKIENGNVKISGNKNLCGVEDINFENLATANGIKVDISNNLGKEECKKLGL']

2. 检查模型可能携带的属性:


In [ ]:
### 📊 Results Leaderboard Generator (已移除)
此单元格已被用户要求清理移除。

🔍 正在扫描文件夹: /content/drive/MyDrive/MPNN/group1 下的所有设计报告...

🎉 成功汇总 14 条设计序列结果！以下为您按结构预测置信度排序后的前 15 个设计：



,Target_Protein,seq,plddt,rmsd,design,mpnn
0,group1,SEEEERRKKEEEERRREEEKREKEREEERRREREREEEEREREEREREEEERRREEEERR/SPRCTGIGEGPFADVKTVNAKKIAVFKGCTVVDGNIYIRAEGFTGSPERNWPPLDPADLALLETIKRITGALTITAWPAGSSGLTPFANLEEILGETRENGEYSLLIEGLTITSIGLTSLKRIKNGNVKISGNKTLCGVENLNWEELGHGEGMKVDVSNNLGAAACAALGL,0.938817,39.028419,0,1.020619
10,test-domain3_2,DEVWARLLALLASHEEAKEALELAEKAVKNRKKAEEALKKGDLETALKYYKESKKALEELRKLVPELPEDVREEARTLLDLALAEEELKLKELEKKIA/EPVCEGIGKGTFANETTINSKNISKFKNCTTINGNINIKKEGFTGSKEKNIAPLDPDKLDLLSSVKKVTGYLKITAWPEGSSGLTPFKNLTEILGETRYNGEYSLLIEGLNITRIGLDNLKLIKNGNVKISGNKNLCGVEDIDWEKIAKGEGIKVDISNNLGKEECKKKGL,0.930528,55.536121,0,0.994671
1,group1,TEAEAAEAAAAAEVAKLLAEAERAAAEGREAEAAALRAEAERVARAGEVKARVLRRKAEEKRRAEEAAR/APVCTGIGFGPFANETTINAKSISSFSGCTTINGNINILASGLTGDPAKNTPPLDPADLSILSSIKSITGYLYITAWPAGENGLTAFKNLEEILGNTKYNGEYSLLITGLNITRIGLDSLRLIENGNVKISGNKNLCGVEDLDWSDYSYGEGIKVDISNNLGKAACAKLGL,0.924958,40.538158,0,1.002136
11,test-domain3_3,EELKQKAAELKAKAEELEAKIEELEKKIEELKKKAEEAAKKGDEELKKKYEKEIEAAKRKKEIYEKKLKELKEEYEKVEKEIEELEK/APVCTGIGYGPYSNETTINEKNISDFLDCTTINGNINIKKEGFEGDKEKNVKPLNPLDLSILSTIKKITGYLYIDAWPKDSNGITPFKNLTEIEGKTKYNGKYSLLVEGLDITSIGFSSLTKIKSGNVRISNNKNLCGVESWNWSSIGTGSGMTVDYSNNLGKEECKKLGL,0.916732,56.723167,0,0.978869
8,test-domain3_0,MSPEEQAKQDALNALVAKAIAAYKEAEKLKAEAAATGDEKLLEKAKELEAKRDELLKKYDEALK/KPKCPGIGTGPFANETTINHRNIKDFENCTEIDGNINIKKEGFEGSEEKKEPPLKPEDLKLLETIEKITGYLKITAWPEGEKGLTPFKNLKEILGKTRYNGKYSLLVEGLNITSLGLNSLEKIVNGNVKISGNKNLCGVESIDWKNIAEGEGIEVDISNNLGEEECKKLGL,0.910042,53.803871,0,0.978233
2,group1,ALEEIAKKLQEALEKEVAKLKADPSLDEEEKAFVERREALAKAIKEAIASGDKEAIKKKLEEYYELLEEELKARKAR/MNVCLGVGYGEFSDEKTVNKKNISSFSNCKMVAGNINILPEGLKGSPEKNWPPLDPEDLSILSSIKTITGYLLITAWPEGSKGLEPFKNLEAILGLTKYNGKYSLVIEGLNITSIGLDSLQLIKNGNVKISGNKNLCGVESLNWSEIGRGEGMTVDVGNNLGKEECAKKGL,0.901849,47.772934,0,1.011103
12,test-domain3_4,PELTPEQLAAARARAAAVRAECDAAGLPAALRRARVQAAIYECRGDAERAAAARAEAAALAAAA/TPICEGVGKGKYADEKTINSKNIADYKNCTVIDGPIRILKEGFEGSPERGVPPLDPADLSILSSIKKITGYLYIDAWPEGENGLTPFKNLEEIEGKELLNGEYSLVIEGLNIKSINLKNLKKIKNGNVKISGNKNLCGVEEIDFSKLAEKEGIKVDISNNLGKEECKKLGL,0.892958,41.695702,0,0.953894
3,group1,KEKEEEEEKKRQELILKKAEKYIKELEEEGKKELAKEYKERLKSSEEERAKAAKEIVKEYEEREEKEEEEKKKLEELKK/KPLCTGVGYGPYKNETSINKKNIKDFKNCTVINGNINILEEGFKGSKEKNVEPLDPKDLDLLKTIKEVTGYLYITAWPEGEKELTPFKNLRKIKGETKYNGKYSLIVEGLNITSLGFDNLKEIENGNVLISGNKNLCGVENLDWSEIGTGPGQEVELENNLGKEECKKLGL,0.871785,48.424145,0,1.021988
4,group1,YNEEKIEEIKKKIKELIETGEETEETLELIEEYEKLLEKKRKEEEERKKKE/KKIVLGIGYGKFKDETTINAKNIKDFENAEIIAGDINIKKEGFTGSKEKNTPPLDPADLSILSTIKKITGALTITAWPEGENGITPFKNLEKILGLTRENGEYSLLIEGLNIKKINLDSLKLIEDGNVKISGNKNLCGVEDWKFEELGEKEGIKVDISNNLGKEECKKLGL,0.850265,45.772038,0,0.906203
9,test-domain3_1,AAAAAAAAAAAEAARRARAEEERRLLERVREELERKEEEERRREEEERLRRELEE/TPVCTGIGYGPYANETTINKKHIADFKNCTTINGNIYIKKEGFEGDPAKNTEPLDPLDLLLLRTIKKITGALYIDAWPEPERELLPFYNLEEILGNTKLNGKYSLLITGLNITSIGLKNLKEIKNGNVRISGNKNLCGVYKIDWSKIAKGEGITVDYSNNLGEEECKKLGL,0.845424,53.239716,0,1.039772


In [ ]:
### 💾 Export Leaderboard Report (已移除)
此单元格已被用户要求清理移除。

SyntaxError: unterminated string literal (detected at line 25) (830113830.py, line 25)